In [ ]:
# Cella 1: Import e Definizione Percorsi
import torch
import os
import glob
from PIL import Image
import numpy as np
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # Usiamo tqdm.notebook per Jupyter
from sklearn.model_selection import train_test_split

# --- IMPOSTAZIONE DEL DEVICE (GPU o CPU) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {device}")

# --- PERCORSI DEL DATASET ---
# Modifica questo flag se necessario
COLAB = False
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_PATH = "/content/drive/MyDrive/Uni/Magistrale/colab/1/ML/Project/training/train"
else:
    # Assicurati che questo percorso sia corretto rispetto alla posizione del tuo notebook
    DATASET_PATH = "../../dataset/training/train" 
    
print(f"Percorso del dataset: {DATASET_PATH}")

In [ ]:
# Cella 2: Analisi del Dataset e Creazione Metadati per lo Split
def analyze_dataset(dataset_path):
    """
    Analizza il dataset per calcolare:
    1. La lista di tutte le directory dei campioni (es. '.../sample_0000').
    2. Il conteggio totale dei pixel per ogni classe (da 0 a 8).
    3. I metadati per la stratificazione (un vettore binario per ogni immagine).
    """
    sample_dirs = sorted(glob.glob(os.path.join(dataset_path, 'sample_*')))
    num_samples = len(sample_dirs)
    print(f"Trovati {num_samples} campioni.")

    if num_samples == 0:
        raise FileNotFoundError(f"Nessun campione trovato nel percorso: {dataset_path}")

    # Assumiamo 9 classi (0-8)
    num_classes = 9
    pixel_counts = np.zeros(num_classes, dtype=np.int64)
    stratify_metadata = []

    print("Inizio analisi del dataset (potrebbe richiedere qualche minuto)...")
    for sample_dir in tqdm(sample_dirs, desc="Analisi campioni"):
        mask_path = os.path.join(sample_dir, 'labels.png')
        
        try:
            mask = Image.open(mask_path)
            mask_np = np.array(mask, dtype=np.uint8)
            
            # 1. Aggiorna conteggio pixel
            counts = np.bincount(mask_np.flatten(), minlength=num_classes)
            pixel_counts += counts
            
            # 2. Crea metadati per la stratificazione
            classes_present = np.unique(mask_np)
            stratify_vector = np.zeros(num_classes, dtype=np.int8)
            stratify_vector[classes_present] = 1
            stratify_metadata.append(stratify_vector)
            
        except FileNotFoundError:
            print(f"Attenzione: maschera non trovata in {sample_dir}. Campione saltato.")
            # Rimuoviamo il campione se la maschera non esiste per mantenere la coerenza
            sample_dirs.remove(sample_dir)
            continue
            
    return sample_dirs, pixel_counts, np.array(stratify_metadata)

# Eseguiamo l'analisi
all_sample_dirs, total_pixel_counts, stratify_data = analyze_dataset(DATASET_PATH)

# --- Verifica dell'analisi ---
print("\n--- Analisi Completata ---")
print(f"Conteggio totale dei pixel per classe:\n{total_pixel_counts}")

# Calcoliamo la frequenza per avere un'idea migliore dello sbilanciamento
total_pixels = total_pixel_counts.sum()
pixel_frequencies = total_pixel_counts / total_pixels
print(f"\nFrequenza dei pixel per classe (%):\n{pixel_frequencies * 100}")

# Visualizziamo la distribuzione
plt.figure(figsize=(12, 6))
plt.bar([f"Classe {i}" for i in range(len(total_pixel_counts))], total_pixel_counts)
plt.title("Distribuzione dei Pixel per Classe (Conteggio Assoluto)")
plt.ylabel("Numero di Pixel")
plt.xlabel("Classe")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"\nForma dei metadati per la stratificazione: {stratify_data.shape}")
print(f"Esempio di metadato per il primo campione: {stratify_data[0]}")

In [ ]:
# Cella 3: Split Stratificato dei Dati

# Definiamo i parametri dello split
VAL_SIZE = 0.2
RANDOM_STATE = 42 # Per la riproducibilità

# Creiamo un identificatore univoco per la stratificazione
# Poiché stratify non accetta array multidimensionali, creiamo una "firma" per ogni riga
# Uniamo i vettori binari in una stringa (es. [1,0,1] -> "101")
stratify_labels = ["".join(map(str, row)) for row in stratify_data]

# Eseguiamo lo split
train_dirs, val_dirs = train_test_split(
    all_sample_dirs,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_labels
)

print(f"Split effettuato:")
print(f"- Campioni di training: {len(train_dirs)}")
print(f"- Campioni di validazione: {len(val_dirs)}")

print("\nEsempio di percorsi:")
print(f"Training dir [0]: {train_dirs[0]}")
print(f"Validation dir [0]: {val_dirs[0]}")

In [ ]:
# Cella 4: Test dello Split - Verifichiamo che la stratificazione abbia funzionato

def check_class_distribution(sample_dirs, num_classes=9):
    """Calcola quante immagini contengono ogni classe in un dato set."""
    class_image_counts = np.zeros(num_classes, dtype=int)
    for sample_dir in sample_dirs:
        mask_path = os.path.join(sample_dir, 'labels.png')
        mask = Image.open(mask_path)
        mask_np = np.array(mask, dtype=np.uint8)
        classes_present = np.unique(mask_np)
        for c in classes_present:
            if c < num_classes:
                class_image_counts[c] += 1
    return class_image_counts

# Calcoliamo la distribuzione per entrambi i set
train_distribution = check_class_distribution(train_dirs)
val_distribution = check_class_distribution(val_dirs)

print("--- Verifica della Distribuzione delle Classi nei Set ---")
print("Numero di IMMAGINI che contengono ogni classe:\n")

print(f"{'Classe':<10} | {'Training Set':<15} | {'Validation Set':<15}")
print("-" * 50)
for i in range(len(train_distribution)):
    # Calcoliamo la percentuale rispetto al totale del set per un confronto equo
    train_perc = (train_distribution[i] / len(train_dirs)) * 100
    val_perc = (val_distribution[i] / len(val_dirs)) * 100
    print(f"Classe {i:<5} | {train_distribution[i]:<5} ({train_perc:4.1f}%)   | {val_distribution[i]:<5} ({val_perc:4.1f}%)")

# Ci aspettiamo che le percentuali (terza e quinta colonna) siano molto simili,
# soprattutto per le classi più rare, a dimostrazione che lo split ha funzionato.

In [ ]:
# --- DEFINIZIONE DELLA CLASSE DATASET ---
import os
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset

class SegmentationDataset(Dataset):
    """
    Versione definitiva e semplificata del Dataset.
    Legge le immagini e le maschere che contengono già gli ID di classe.
    """
    def __init__(self, sample_dirs, transform=None):
        self.sample_dirs = sample_dirs
        self.transform = transform

    def __len__(self):
        return len(self.sample_dirs)

    def __getitem__(self, idx):
        sample_dir = self.sample_dirs[idx]

        img_path = os.path.join(sample_dir, 'rgb.jpg')
        mask_path = os.path.join(sample_dir, 'labels.png')

        # Carichiamo l'immagine RGB e la maschera
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path) # È già in scala di grigi con indici 0-8

        # Convertiamo in array NumPy
        # Normalizziamo l'immagine a [0, 1] per la stabilità del training
        image_np = np.array(image, dtype=np.float32) / 255.0
        # Le maschere contengono ID, quindi usiamo un tipo intero
        mask_np = np.array(mask, dtype=np.int64)

        if self.transform:
            # Qui applicheremo la data augmentation nelle fasi successive
            pass

        # Convertiamo in tensori PyTorch
        # Immagine -> [C, H, W]
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1)
        # Maschera -> [H, W] (la loss CrossEntropy vuole questo formato)
        mask_tensor = torch.from_numpy(mask_np)

        return image_tensor, mask_tensor

In [ ]:
from torch.utils.data import DataLoader

# Creiamo le istanze del dataset
train_dataset = SegmentationDataset(sample_dirs=train_dirs, transform=None)
val_dataset = SegmentationDataset(sample_dirs=val_dirs, transform=None)

# Creiamo i DataLoader
BATCH_SIZE = 8 # Esempio, da definire in base alla VRAM
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print("\n--- Dataset e DataLoader creati ---")
print(f"Numero di batch nel training loader: {len(train_loader)}")
print(f"Numero di batch nel validation loader: {len(val_loader)}")

# Testiamo il caricamento di un batch
images, masks = next(iter(train_loader))
print("\nVerifica di un batch:")
print(f"Forma del tensore delle immagini: {images.shape}") # Atteso: [BATCH_SIZE, 3, H, W]
print(f"Forma del tensore delle maschere: {masks.shape}")   # Atteso: [BATCH_SIZE, H, W]
print(f"Tipo di dato immagini: {images.dtype}")
print(f"Tipo di dato maschere: {masks.dtype}")